In [4]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import math

def create_double_torus(R1=2, R2=2, r=0.5, n_theta=100, n_phi=50, distance=3):
    """
    Create a double torus with two tori of radii R1 and R2, tube radius r,
    separated by distance.

    Parameters:
    - R1, R2: major radii of the two tori
    - r: minor radius (tube radius) of both tori
    - n_theta, n_phi: resolution parameters
    - distance: distance between the centers of the two tori

    Returns:
    - x, y, z: coordinates of the double torus
    - theta_vals, phi_vals: parameter values for mapping
    """
    theta = np.linspace(0, 2*np.pi, n_theta)
    phi = np.linspace(0, 2*np.pi, n_phi)
    theta_grid, phi_grid = np.meshgrid(theta, phi)

    # First torus
    x1 = (R1 + r*np.cos(phi_grid)) * np.cos(theta_grid) - distance/2
    y1 = (R1 + r*np.cos(phi_grid)) * np.sin(theta_grid)
    z1 = r * np.sin(phi_grid)

    # Second torus
    x2 = (R2 + r*np.cos(phi_grid)) * np.cos(theta_grid) + distance/2
    y2 = (R2 + r*np.cos(phi_grid)) * np.sin(theta_grid)
    z2 = r * np.sin(phi_grid)

    return x1, y1, z1, x2, y2, z2, theta_grid, phi_grid

def create_logarithmic_spiral(R, r, n_turns=3, n_points=1000, a=0.1, b=0.1):
    """
    Create a logarithmic spiral on a torus.

    Parameters:
    - R: major radius of the torus
    - r: minor radius (tube radius) of the torus
    - n_turns: number of turns the spiral makes around the torus
    - n_points: number of points to generate
    - a, b: parameters controlling the spiral's growth rate

    Returns:
    - x, y, z: coordinates of the spiral
    """
    t = np.linspace(0, n_turns * 2 * np.pi, n_points)

    # Logarithmic spiral parameters
    rho = r * np.exp(a * t)  # Radius grows exponentially

    # Ensure rho doesn't exceed the tube radius
    rho = np.minimum(rho, r * 0.95)

    # Angular position around the major circle
    theta = t

    # Angular position around the minor circle
    phi = b * t

    # Convert to Cartesian coordinates
    x = (R + rho * np.cos(phi)) * np.cos(theta)
    y = (R + rho * np.cos(phi)) * np.sin(theta)
    z = rho * np.sin(phi)

    return x, y, z, t

def create_mass_points(masses, lambda_val=0.9931, omega=3.883, epsilon=0.309, K=1e-3):
    """
    Create points representing particle masses on the spiral.

    Parameters:
    - masses: dictionary of particle names and their masses in GeV
    - lambda_val, omega, epsilon, K: parameters of the spiral model

    Returns:
    - positions: dictionary of particle names and their (rho, theta) positions
    """
    positions = {}

    for name, mass in masses.items():
        # Generation number (approximated)
        if "1" in name:
            n = 0
        elif "2" in name:
            n = 1
        elif "3" in name:
            n = 2
        else:
            n = 0

        # Hypercharge
        if "u" in name or "c" in name or "t" in name:
            Y = 1
        elif "d" in name or "s" in name or "b" in name:
            Y = -1
        else:  # leptons
            Y = 0

        # Calculate position using the spiral equation
        log_mass = np.log10(mass)
        theta = n * 2*np.pi + epsilon * np.sin(omega * n + Y * np.pi/3)
        rho = (log_mass - np.log10(K * (2**Y))) / lambda_val

        positions[name] = (rho, theta)

    return positions

def map_positions_to_torus(positions, R, r):
    """
    Map (rho, theta) positions to 3D coordinates on a torus.

    Parameters:
    - positions: dictionary of particle names and their (rho, theta) positions
    - R: major radius of the torus
    - r: minor radius (tube radius) of the torus

    Returns:
    - coords: dictionary of particle names and their (x, y, z) coordinates
    """
    coords = {}

    for name, (rho_norm, theta) in positions.items():
        # Normalize rho to be within the tube radius
        rho = r * 0.9 * (rho_norm / 3)  # Assuming max rho_norm is about 3

        # Angular position around the minor circle (arbitrary for visualization)
        phi = theta % (2*np.pi)

        # Convert to Cartesian coordinates
        x = (R + rho * np.cos(phi)) * np.cos(theta)
        y = (R + rho * np.cos(phi)) * np.sin(theta)
        z = rho * np.sin(phi)

        coords[name] = (x, y, z)

    return coords

def create_plotly_double_torus_visualization():
    """
    Create an interactive Plotly visualization of the double torus with spiral and particle masses.
    """
    # Create the double torus
    R = 2
    r = 0.5
    distance = 3
    x1, y1, z1, x2, y2, z2, theta_grid, phi_grid = create_double_torus(R1=R, R2=R, r=r, distance=distance)

    # Create logarithmic spirals on both tori
    spiral_x1, spiral_y1, spiral_z1, t1 = create_logarithmic_spiral(R, r, n_turns=3, a=0.1, b=0.1)
    spiral_x2, spiral_y2, spiral_z2, t2 = create_logarithmic_spiral(R, r, n_turns=3, a=0.1, b=0.1)

    # Shift the second spiral to the second torus
    spiral_x2 += distance

    # Define particle masses (in GeV)
    quark_masses = {
        "u1": 0.0022,  # up quark
        "d1": 0.0047,  # down quark
        "c2": 1.27,    # charm quark
        "s2": 0.093,   # strange quark
        "t3": 173,     # top quark
        "b3": 4.18     # bottom quark
    }

    lepton_masses = {
        "e1": 0.000511,  # electron
        "mu2": 0.1057,   # muon
        "tau3": 1.777    # tau
    }

    neutrino_masses = {
        "nu_e1": 2.1e-12,     # electron neutrino (in GeV, ~2.1 meV)
        "nu_mu2": 8.9e-12,    # muon neutrino (in GeV, ~8.9 meV)
        "nu_tau3": 5.03e-11   # tau neutrino (in GeV, ~50.3 meV)
    }

    # Calculate positions
    up_type_positions = create_mass_points({k: v for k, v in quark_masses.items() if k[0] in ['u', 'c', 't']})
    down_type_positions = create_mass_points({k: v for k, v in quark_masses.items() if k[0] in ['d', 's', 'b']})
    lepton_positions = create_mass_points(lepton_masses)
    neutrino_positions = create_mass_points(neutrino_masses)

    # Map to 3D coordinates
    up_type_coords = map_positions_to_torus(up_type_positions, R, r)
    # Shift to first torus
    up_type_coords = {k: (v[0]-distance/2, v[1], v[2]) for k, v in up_type_coords.items()}

    down_type_coords = map_positions_to_torus(down_type_positions, R, r)
    # Shift to second torus
    down_type_coords = {k: (v[0]+distance/2, v[1], v[2]) for k, v in down_type_coords.items()}

    lepton_coords = map_positions_to_torus(lepton_positions, R, r)
    # Place at junction
    lepton_coords = {k: (v[0], v[1], v[2]) for k, v in lepton_coords.items()}

    neutrino_coords = map_positions_to_torus(neutrino_positions, R, r)
    # Place at junction with offset
    neutrino_coords = {k: (v[0], v[1], v[2]+0.5) for k, v in neutrino_coords.items()}

    # Create the figure
    fig = go.Figure()

    # Add the first torus
    fig.add_trace(go.Surface(
        x=x1, y=y1, z=z1,
        colorscale='Blues',
        opacity=0.7,
        showscale=False,
        name="First Torus (Up-type)"
    ))

    # Add the second torus
    fig.add_trace(go.Surface(
        x=x2, y=y2, z=z2,
        colorscale='Reds',
        opacity=0.7,
        showscale=False,
        name="Second Torus (Down-type)"
    ))

    # Add the first spiral
    fig.add_trace(go.Scatter3d(
        x=spiral_x1 - distance/2,
        y=spiral_y1,
        z=spiral_z1,
        mode='lines',
        line=dict(color='darkblue', width=5),
        name="Up-type Spiral"
    ))

    # Add the second spiral
    fig.add_trace(go.Scatter3d(
        x=spiral_x2 - distance/2,
        y=spiral_y2,
        z=spiral_z2,
        mode='lines',
        line=dict(color='darkred', width=5),
        name="Down-type Spiral"
    ))

    # Add up-type quarks
    for name, (x, y, z) in up_type_coords.items():
        fig.add_trace(go.Scatter3d(
            x=[x], y=[y], z=[z],
            mode='markers+text',
            marker=dict(size=10, color='blue'),
            text=[name],
            name=name
        ))

    # Add down-type quarks
    for name, (x, y, z) in down_type_coords.items():
        fig.add_trace(go.Scatter3d(
            x=[x], y=[y], z=[z],
            mode='markers+text',
            marker=dict(size=10, color='red'),
            text=[name],
            name=name
        ))

    # Add leptons
    for name, (x, y, z) in lepton_coords.items():
        fig.add_trace(go.Scatter3d(
            x=[x], y=[y], z=[z],
            mode='markers+text',
            marker=dict(size=10, color='green'),
            text=[name],
            name=name
        ))

    # Add neutrinos
    for name, (x, y, z) in neutrino_coords.items():
        fig.add_trace(go.Scatter3d(
            x=[x], y=[y], z=[z],
            mode='markers+text',
            marker=dict(size=8, color='purple'),
            text=[name],
            name=name
        ))

    # Update layout
    fig.update_layout(
        title="Holographic Double-Torus Model with Logarithmic Spiral and Particle Masses",
        scene=dict(
            xaxis_title="X",
            yaxis_title="Y",
            zaxis_title="Z",
            aspectmode='data'
        ),
        legend=dict(
            yanchor="top",
            y=0.99,
            xanchor="left",
            x=0.01
        ),
        margin=dict(l=0, r=0, b=0, t=30),
        scene_camera=dict(
            eye=dict(x=1.5, y=1.5, z=1.5)
        )
    )

    # Save the figure
    fig.write_html("/home/ubuntu/visualizations/double_torus_spiral_3d.html")

    return fig

# Create the visualization
fig = create_plotly_double_torus_visualization()

# Also create a static version for LaTeX
def create_matplotlib_double_torus_visualization():
    """
    Create a static Matplotlib visualization of the double torus with spiral.
    """
    # Create the double torus
    R = 2
    r = 0.5
    distance = 3
    x1, y1, z1, x2, y2, z2, theta_grid, phi_grid = create_double_torus(R1=R, R2=R, r=r, distance=distance)

    # Create logarithmic spirals on both tori
    spiral_x1, spiral_y1, spiral_z1, t1 = create_logarithmic_spiral(R, r, n_turns=3, a=0.1, b=0.1)
    spiral_x2, spiral_y2, spiral_z2, t2 = create_logarithmic_spiral(R, r, n_turns=3, a=0.1, b=0.1)

    # Shift the second spiral to the second torus
    spiral_x2 += distance

    # Create the figure
    fig = plt.figure(figsize=(12, 10))
    ax = fig.add_subplot(111, projection='3d')

    # Plot the first torus
    ax.plot_surface(x1, y1, z1, color='blue', alpha=0.3)

    # Plot the second torus
    ax.plot_surface(x2, y2, z2, color='red', alpha=0.3)

    # Plot the first spiral
    ax.plot(spiral_x1 - distance/2, spiral_y1, spiral_z1, color='darkblue', linewidth=2)

    # Plot the second spiral
    ax.plot(spiral_x2 - distance/2, spiral_y2, spiral_z2, color='darkred', linewidth=2)

    # Set labels and title
    ax.set_xlabel('X')
    ax.set_ylabel('Y')
    ax.set_zlabel('Z')
    ax.set_title('Holographic Double-Torus Model with Logarithmic Spiral')

    # Set the viewing angle
    ax.view_init(elev=30, azim=45)

    # Save the figure
    plt.savefig("/home/ubuntu/visualizations/double_torus_spiral_3d.png", dpi=300, bbox_inches='tight')
    plt.close()

# Create the static visualization
create_matplotlib_double_torus_visualization()

print("Visualizations created successfully!")


FileNotFoundError: [Errno 2] No such file or directory: '/home/ubuntu/visualizations/double_torus_spiral_3d.html'